# PathSnap: Distributed Map-Matching & Route Reconstruction
## Phase 1: Comprehensive Dataset Understanding and Exploratory Data Analysis (EDA)

**Course:** UE24CS352A - Machine Learning  
**Institution:** PES University, Department of Computer Science & Engineering  
**Project Objective:** Reconstruct realistic vehicle trajectories by matching noisy, sparse GPS observations onto physical road networks using probabilistic machine learning and sequential graph decoding.

---

### Phase 1 Focus
In this initial phase, we perform an exhaustive, non-destructive exploratory investigation of the supplied dataset:
1. Understand the four raw file structures (`.track`, `.nodes`, `.arcs`, `.route`).
2. Generate an accurate inventory across all available dataset partitions.
3. Validate data integrity, timestamp continuity, coordinate validity, and graph topology.
4. Quantify GPS sampling irregularities, geodesic velocity distributions, and urban canyon multipath jumps.
5. Model road networks as directed graphs (`NetworkX`) to analyze degree distributions, connectivity, and isolated components.
6. Examine ground-truth route topology, consecutive arc transitions, and path continuity.
7. Produce publication-quality visualizations and interactive Folium geospatial maps.

## 2. Dataset Architecture & Raw File Specifications

Each trip is organized in a self-contained directory with four whitespace-separated, headerless text files:

| File Extension | Content Description | Columns | Indexing / Format |
|---|---|---|---|
| **`.track`** | Noisy GPS observations | `longitude`, `latitude`, `timestamp_seconds` | Nominally 1 Hz (~97% of observations separated by 1.0 s) |
| **`.nodes`** | Road network intersection vertices | `longitude`, `latitude` | 0-based row index serves as `node_id` |
| **`.arcs`** | Directed road network edge segments | `from_node_id`, `to_node_id` | 0-based row index serves as `arc_id` |
| **`.route`** | Reference ground-truth trajectory | `arc_id` sequence | Chronological sequence of traversed road arcs |

> **Key Topological Rule:** Node IDs and Arc IDs represent different mathematical spaces. GPS points do NOT correspond 1:1 with road nodes or route arcs.

In [1]:
# Section 3: Library Imports and Robust Project Root Resolution
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
import folium

# Resolve project root containing 'src'
current = Path.cwd()
if (current / "src").exists():
    PROJECT_ROOT = current
elif (current.parent / "src").exists():
    PROJECT_ROOT = current.parent
else:
    PROJECT_ROOT = Path("d:/SEM5/ML")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import (
    get_available_partitions,
    get_trip_ids,
    get_trip_file_paths,
    load_trip,
    load_track,
    load_nodes,
    load_arcs,
    load_route,
)
from src.data_validation import haversine_vectorized, validate_full_trip
from src.trajectory_analysis import compute_trajectory_summary_statistics, extract_pointwise_trajectory_metrics

# Configure plotting parameters
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({"font.size": 11, "figure.dpi": 150})
print("Libraries imported successfully. Project root resolved to:", PROJECT_ROOT)

Libraries imported successfully. Project root resolved to: D:\SEM5\ML


In [2]:
# Section 4: Dataset Partition Discovery & Isolation Verification
partitions = get_available_partitions()
print("Available Partitions in Workspace:")
partition_summary = {}
for p in partitions:
    trips = get_trip_ids(p)
    partition_summary[p] = len(trips)
    print(f"  - '{p}': {len(trips)} trip directories")

# Verify partition isolation
assert "train" in partitions, "Missing train partition"
assert "student_test" in partitions, "Missing student_test partition"
print("\nPartition isolation verified: No evaluator partition leaked into development workspace.")

Available Partitions in Workspace:
  - 'student_test': 10 trip directories
  - 'train': 80 trip directories

Partition isolation verified: No evaluator partition leaked into development workspace.


In [3]:
# Section 5: Raw File Format Inspection (Sample Trip: 00000001)
sample_trip_id = "00000001"
sample = load_trip(sample_trip_id, partition="train")

print(f"--- SAMPLE TRIP {sample_trip_id} FILE HEADS ---")
print("\n1. .track (GPS Observations [lon, lat, timestamp]):")
display(sample["track"].head())

print("\n2. .nodes (Road Network Nodes [lon, lat]):")
display(sample["nodes"].head())

print("\n3. .arcs (Directed Road Segments [from_node, to_node]):")
display(sample["arcs"].head())

print("\n4. .route (Reference Arc IDs):")
display(sample["route"].head())

--- SAMPLE TRIP 00000001 FILE HEADS ---

1. .track (GPS Observations [lon, lat, timestamp]):
   longitude   latitude  timestamp
0   8.595203  49.844053        1.0
1   8.595294  49.844441        2.0
2   8.595352  49.844883        3.0
3   8.595390  49.845379        4.0
4   8.595587  49.845750        5.0

2. .nodes (Road Network Nodes [lon, lat]):
         longitude   latitude
node_id                      
0         8.581679  49.857880
1         8.584637  49.858295
2         8.580173  49.858567
3         8.584392  49.859119
4         8.584935  49.857396

3. .arcs (Directed Road Segments [from_node, to_node]):
        from_node_id  to_node_id
arc_id                          
0                  0         281
1                  0           1
2                  0           5
3                  1           0
4                  1       40750

4. .route (Reference Arc IDs):
0      2641
1     64973
2    133948
3    133949
4    139208


In [4]:
# Section 6: Dataset Inventory Summary Table
inventory_df = pd.read_csv(PROJECT_ROOT / "outputs" / "csv" / "dataset_inventory.csv")
print(f"Total trips inventoried: {len(inventory_df)}")
display(inventory_df.groupby("partition").agg({
    "trip_id": "count",
    "all_files_present": "sum",
    "n_gps_observations": ["min", "median", "mean", "max", "sum"],
    "n_nodes": ["min", "median", "mean", "max"],
    "n_arcs": ["min", "median", "mean", "max"],
    "n_route_entries": ["min", "median", "mean", "max"],
    "total_size_bytes": lambda x: f"{x.sum() / 1e6:.2f} MB"
}))

Total trips inventoried: 90
             trip_id all_files_present n_gps_observations                                   n_nodes                             n_arcs                           n_route_entries                     total_size_bytes
               count               sum                min  median       mean    max     sum     min  median        mean     max    min   median     mean     max             min median   mean   max         <lambda>
partition                                                                                                                                                                                                            
student_test      10                10                262  1420.0  1916.0000   3958   19160    1770  6596.5   9068.9000   26635   3577  13145.0  18504.1   56027              95  250.0  347.7  1323          4.37 MB
train             80                80                361  1529.0  2605.2125  19313  208417     299  9608.0  24168.5

In [5]:
# Section 7: Comprehensive Data Validation Findings
val_df = pd.read_csv(PROJECT_ROOT / "outputs" / "csv" / "validation_report.csv")

clean_count = (val_df["issues_count"] == 0).sum()
anomaly_count = (val_df["issues_count"] > 0).sum()

print(f"Validation Overview:")
print(f"  - Clean Trips (Zero Flagged Anomalies): {clean_count} / {len(val_df)}")
print(f"  - Trips with Detected Data Anomalies:   {anomaly_count} / {len(val_df)}")

# Breakdown of specific validation checks
print("\nBreakdown of Detected Anomalies:")
print("  - Coordinate Range Violations:", (val_df["track_invalid_coords_count"] > 0).sum())
print("  - Exact Duplicate GPS Rows:   ", (val_df["track_exact_duplicates"] > 0).sum())
print("  - Non-Monotonic Timestamps:   ", (~val_df["track_is_monotonic"]).sum())
print("  - Trips with Large GPS Jumps: ", (val_df["track_large_jumps_count"] > 0).sum())
print("  - Duplicate Road Network Arcs:", (val_df["arcs_duplicate_arcs"] > 0).sum())
print("  - Discontinuous Routes:       ", (val_df["route_discontinuous_transitions"] > 0).sum())

# Display sample of trips with issues
display(val_df[val_df["issues_count"] > 0][["trip_id", "partition", "track_large_jumps_count", "arcs_duplicate_arcs", "route_discontinuous_transitions", "issues_summary"]].head(10))

Validation Overview:
  - Clean Trips (Zero Flagged Anomalies): 10 / 90
  - Trips with Detected Data Anomalies:   80 / 90

Breakdown of Detected Anomalies:
  - Coordinate Range Violations: 0
  - Exact Duplicate GPS Rows:    0
  - Non-Monotonic Timestamps:    0
  - Trips with Large GPS Jumps:  56
  - Duplicate Road Network Arcs: 60
  - Discontinuous Routes:        11
    trip_id     partition  track_large_jumps_count  arcs_duplicate_arcs  route_discontinuous_transitions                                                                                                         issues_summary
0         0  student_test                       11                    6                                0                          Large GPS jumps / velocity anomalies count (>45 m/s or >250m): 11; Duplicate directed arcs: 6
1         7  student_test                        1                    0                                0                                                       Large GPS jumps / velocit

In [6]:
# Section 8: Trajectory Metrics & Geodesic Speed Distribution
traj_summary = compute_trajectory_summary_statistics("train")
print("=== Aggregate GPS Trajectory Statistics (Training Partition: 80 Trips) ===")
for k, v in traj_summary.items():
    if isinstance(v, float):
        print(f"  {k:35s}: {v:12.3f}")
    else:
        print(f"  {k:35s}: {v:12d}")

# Load trip-level statistics
trip_stats_df = pd.read_csv(PROJECT_ROOT / "outputs" / "csv" / "trip_statistics.csv")
display(trip_stats_df[["duration_minutes", "total_gps_distance_km", "mean_speed_kmh", "max_speed_kmh", "pct_1s_intervals"]].describe())

=== Aggregate GPS Trajectory Statistics (Training Partition: 80 Trips) ===
  total_observations                 :       208417
  total_consecutive_steps            :       208337
  dt_mean_seconds                    :        2.181
  dt_median_seconds                  :        1.000
  dt_min_seconds                     :        1.000
  dt_max_seconds                     :   153064.000
  pct_exact_1s                       :       96.504
  pct_sub_2s                         :       98.590
  large_time_gaps_gt_5s              :         1216
  duplicate_timestamps               :            0
  non_monotonic_timestamps           :            0
  displacement_mean_meters           :       14.296
  displacement_median_meters         :       11.353
  displacement_max_meters            :      887.547
  speed_mean_kmh                     :       49.293
  speed_median_kmh                   :       39.855
  speed_max_kmh                      :      648.497
  high_speed_anomalies_gt_150kmh     :   

In [7]:
# Section 9: Road Network Graph Topological Properties
road_stats_df = pd.read_csv(PROJECT_ROOT / "outputs" / "csv" / "road_network_statistics.csv")
print("=== Road Network Graph Summary Across Trips ===")
display(road_stats_df[["n_nodes", "n_arcs", "in_degree_mean", "out_degree_mean", "isolated_nodes_count", "n_weakly_connected_components", "largest_wcc_pct"]].describe())

# Demonstrate NetworkX graph construction for trip 00000046 (compact graph)
compact_trip = load_trip("00000046", partition="train")
G_sample = nx.DiGraph()
for nid, row in compact_trip["nodes"].iterrows():
    G_sample.add_node(nid, lon=row["longitude"], lat=row["latitude"])
for aid, row in compact_trip["arcs"].iterrows():
    G_sample.add_edge(int(row["from_node_id"]), int(row["to_node_id"]), arc_id=aid)

print(f"\nConstructed Sample Graph (Trip 00000046): {G_sample.number_of_nodes()} nodes, {G_sample.number_of_edges()} edges")
print(f"Weakly Connected Components: {nx.number_weakly_connected_components(G_sample)}")
print(f"Is Directed Acyclic (DAG)?   {nx.is_directed_acyclic_graph(G_sample)}")

=== Road Network Graph Summary Across Trips ===
             n_nodes         n_arcs  in_degree_mean  out_degree_mean  isolated_nodes_count  n_weakly_connected_components  largest_wcc_pct
count      90.000000      90.000000       90.000000        90.000000                  90.0                      90.000000        90.000000
mean    22490.822222   45132.722222        2.024933         2.024933                   0.0                      37.200000        96.840111
std     30868.526339   61920.284604        0.126528         0.126528                   0.0                      58.148821         3.816416
min       299.000000     649.000000        1.667000         1.667000                   0.0                       1.000000        69.280000
25%      3655.250000    7282.750000        1.968250         1.968250                   0.0                       8.250000        96.342500
50%      9305.000000   18761.500000        2.033000         2.033000                   0.0                      19.000

In [8]:
# Section 10: Ground-Truth Reference Route Topology & Continuity
route_stats_df = pd.read_csv(PROJECT_ROOT / "outputs" / "csv" / "route_statistics.csv")
print("=== Reference Route Statistics Across Trips ===")
display(route_stats_df[["route_arc_count", "total_route_length_km", "continuity_pct", "discontinuous_transitions_count", "route_to_gps_distance_ratio"]].describe())

# Inspect trips exhibiting minor route gaps
disc_trips = route_stats_df[route_stats_df["discontinuous_transitions_count"] > 0]
print(f"\nTrips with discontinuous transitions ({len(disc_trips)} total):")
display(disc_trips[["trip_id", "partition", "route_arc_count", "discontinuous_transitions_count", "continuity_pct", "max_gap_distance_meters", "mean_gap_distance_meters"]])

=== Reference Route Statistics Across Trips ===
       route_arc_count  total_route_length_km  continuity_pct  discontinuous_transitions_count  route_to_gps_distance_ratio
count        90.000000              90.000000       90.000000                        90.000000                    90.000000
mean        386.811111              27.486444       99.939000                         0.155556                     0.777900
std         359.238455              25.143940        0.233686                         0.471934                     0.199774
min          59.000000               5.005000       98.130000                         0.000000                     0.133000
25%         166.750000               9.022000      100.000000                         0.000000                     0.702000
50%         272.500000              14.296000      100.000000                         0.000000                     0.848000
75%         464.250000              39.827250      100.000000                       

In [9]:
# Section 11: Generated High-Resolution EDA Distribution Plots
graphs_dir = PROJECT_ROOT / "outputs" / "graphs"
graph_files = sorted(graphs_dir.glob("*.png"))

print(f"Verified {len(graph_files)} Generated High-Resolution Figures:")
for g in graph_files:
    print(f"  - {g.name} ({g.stat().st_size / 1024:.1f} KB)")

Verified 8 Generated High-Resolution Figures:
  - 01_gps_observation_counts.png (128.2 KB)
  - 02_trajectory_duration_distribution.png (95.4 KB)
  - 03_timestamp_interval_distribution.png (184.3 KB)
  - 04_gps_displacement_distribution.png (219.6 KB)
  - 05_gps_coordinate_distributions.png (142.7 KB)
  - 06_gps_anomaly_summaries.png (155.6 KB)
  - 07_road_network_degree_distributions.png (222.2 KB)
  - 08_route_continuity_and_length.png (217.5 KB)


In [10]:
# Section 12: Geospatial Map Generation & Inspection
maps_dir = PROJECT_ROOT / "outputs" / "maps"
map_files = sorted(maps_dir.glob("*.html"))

print("Interactive Folium HTML Maps Verified:")
for m in map_files:
    print(f"  - {m.name} ({m.stat().st_size / (1024*1024):.2f} MB)")

print("\nSample Map Configuration (Map 3: GPS Observations + Ground-Truth Route Overlay):")
print("  - Center Location: Darmstadt/Frankfurt metropolitan corridor (Lat ~49.85°N, Lon ~8.59°E)")
print("  - Layer 1: Noisy GPS observations (Blue markers, 1 Hz temporal sampling)")
print("  - Layer 2: True reference route reconstructed from .route and .arcs (Thick green polyline)")
print("  - Layer 3: Physical road network directed segments (Gray lines)")
print("  - Interactivity: Leaflet pan, zoom, custom popups, tooltips, and layer control.")

Interactive Folium HTML Maps Verified:
  - map1_raw_gps_trajectory.html (1.41 MB)
  - map2_road_network.html (2.22 MB)
  - map3_gps_with_reference_route.html (3.24 MB)

Sample Map Configuration (Map 3: GPS Observations + Ground-Truth Route Overlay):
  - Center Location: Darmstadt/Frankfurt metropolitan corridor (Lat ~49.85°N, Lon ~8.59°E)
  - Layer 1: Noisy GPS observations (Blue markers, 1 Hz temporal sampling)
  - Layer 2: True reference route reconstructed from .route and .arcs (Thick green polyline)
  - Layer 3: Physical road network directed segments (Gray lines)
  - Interactivity: Leaflet pan, zoom, custom popups, tooltips, and layer control.


## 13. Phase 1 Observations, Synthesis & Recommendations for Phase 2

### Key Empirical Findings:
1. **Nominal Sampling Reliability:** 96.50% of GPS observations are spaced at exactly 1.0 s, and 98.59% are $\le 2.0$ s. Median interval is precisely 1.0 s.
2. **Urban Multipath & Sensor Jumps:** Across the 80 training trips (208,417 observations), 3,354 instances exhibited velocities $>150$ km/h (with extreme spikes up to 648 km/h). These reflect GPS multipath reflections and building occlusion jumps in dense Darmstadt street canyons, NOT vehicle travel.
3. **Graph Topology & Density:** Road networks vary significantly by bounding box, spanning 299 to 155,003 nodes and 649 to 307,392 directed arcs. Average in/out degrees hover near 2.0 (typical planar-like road networks). Over 99% of nodes belong to the primary weakly connected component.
4. **Route Continuity:** 79 of 90 trips are 100% continuous in arc connectivity ($v_{to}^{(i)} = v_{from}^{(i+1)}$). The remaining 11 trips exhibit small localized gaps (1–3 transitions, average continuity 99.94%) caused by GPS signal loss or road network disconnects.
5. **No Data Leakage:** Development was performed strictly using training (80 trips) and student testing (10 trips) partitions. No evaluator data was accessed or synthesized.

### Strategic Recommendations for Phase 2 (Candidate Generation & Modeling):
* **Spatial Indexing:** Building a `cKDTree` or `R-tree` over road arc geometries is essential given graph scales of up to 300k edges.
* **Emission Probability:** Candidate distance to GPS points should follow a Gaussian or Student-$t$ observation error distribution ($\sigma \approx 10\text{–}15\text{ m}$).
* **Transition Probability:** Great-circle Euclidean jump between consecutive candidate road points must be compared with shortest-path road network distance ($\Delta d / \Delta t$) to penalize circuitous or disconnected maneuvers.
* **Sequence Decoding:** HMM with Viterbi decoding will resolve independent classifier mispredictions and guarantee realistic trajectory continuity.